# EuroSAT Satellite Image Classification
## Final Course-Level Capstone Version

**Student:** Jianye Chen  
**Dataset:** EuroSAT  
**Framework:** PyTorch

This notebook uses course-level PyTorch methods to build and evaluate a 10-class satellite image classifier. The final solution uses a 3-layer CNN, simple data augmentation, validation-loss early stopping, manual hyperparameter comparison, and a frozen ResNet18 comparison.

## 1. Imports and Settings

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data"
IMAGE_SIZE = 64
TUNING_EPOCHS = 6
FINAL_EPOCHS = 15
RESNET_EPOCHS = 6
PATIENCE = 3
print("Device:", device)

## 2. Final Data Augmentation

The final training transform intentionally stays simple. `RandomResizedCrop` is restricted to `scale=(0.8, 1.0)` so that most of the satellite image remains visible. Random horizontal flipping is retained. Additional VerticalFlip, Rotation, and ColorJitter were tested later but did not improve the final result.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

## 3. Load and Split EuroSAT

EuroSAT contains 27,000 RGB satellite images in 10 land-use classes. A stratified 70% / 15% / 15% train-validation-test split is used.

In [ ]:
base_dataset = datasets.EuroSAT(root=DATA_DIR, download=True)
class_names = base_dataset.classes
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

train_indices, temp_indices = train_test_split(
    indices, test_size=0.30, random_state=random_seed, stratify=targets
)
val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.50, random_state=random_seed, stratify=targets[temp_indices]
)

train_full = datasets.EuroSAT(root=DATA_DIR, transform=train_transform, download=False)
eval_full = datasets.EuroSAT(root=DATA_DIR, transform=test_transform, download=False)
train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(eval_full, val_indices)
test_dataset = Subset(eval_full, test_indices)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print(class_names)

In [ ]:
def make_loaders(batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader

## 4. Custom CNN

The network uses three convolution layers with BatchNorm, ReLU, MaxPool, Dropout, and fully connected layers, following the CNN methods used in class.

In [ ]:
class EuroSATCNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(EuroSATCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

## 5. Training and Early Stopping

In [ ]:
def train_model(model, train_loader, val_loader, learning_rate, epochs, patience, save_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_val_acc = 0
    counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = correct = total = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.size(0)
            correct += (output.argmax(1) == target).sum().item()
            total += target.size(0)
        train_loss = total_loss / total
        train_acc = correct / total

        model.eval()
        val_loss_sum = val_correct = val_total = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                val_loss_sum += loss.item() * data.size(0)
                val_correct += (output.argmax(1) == target).sum().item()
                val_total += target.size(0)
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping")
                break

    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, history, best_val_loss, best_val_acc

## 6. Hyperparameter Tuning

Three manual settings are compared for six epochs. The final setting is selected using the **lowest validation loss**.

In [ ]:
experiments = [
    {"name":"Experiment 1", "learning_rate":0.001, "batch_size":32, "dropout":0.3},
    {"name":"Experiment 2", "learning_rate":0.0005, "batch_size":32, "dropout":0.3},
    {"name":"Experiment 3", "learning_rate":0.001, "batch_size":64, "dropout":0.5}
]

tuning_results = []
for exp in experiments:
    torch.manual_seed(random_seed)
    train_loader, val_loader, _ = make_loaders(exp["batch_size"])
    model = EuroSATCNN(exp["dropout"]).to(device)
    model, history, best_loss, best_acc = train_model(
        model, train_loader, val_loader, exp["learning_rate"],
        TUNING_EPOCHS, PATIENCE, "tuning_" + exp["name"].replace(" ", "_") + ".pth"
    )
    tuning_results.append({**exp, "best_val_loss":best_loss, "val_accuracy_at_best_loss":best_acc})

tuning_df = pd.DataFrame(tuning_results).sort_values("best_val_loss").reset_index(drop=True)
tuning_df

### Recorded Tuning Result

| Experiment | Learning Rate | Batch Size | Dropout | Best Validation Loss | Validation Accuracy at Best Loss |
|---|---:|---:|---:|---:|---:|
| **Experiment 2** | **0.0005** | **32** | **0.3** | **0.4457** | **84.12%** |
| Experiment 1 | 0.0010 | 32 | 0.3 | 0.4675 | 83.51% |
| Experiment 3 | 0.0010 | 64 | 0.5 | 0.6175 | 78.54% |

Experiment 2 was selected because it achieved the lowest validation loss.

## 7. Train the Final Custom CNN

In [ ]:
best = tuning_df.iloc[0]
best_lr = float(best["learning_rate"])
best_batch = int(best["batch_size"])
best_dropout = float(best["dropout"])

train_loader, val_loader, test_loader = make_loaders(best_batch)
torch.manual_seed(random_seed)
final_cnn = EuroSATCNN(best_dropout).to(device)
final_cnn, cnn_history, cnn_best_loss, cnn_best_val_acc = train_model(
    final_cnn, train_loader, val_loader, best_lr, FINAL_EPOCHS, PATIENCE, "final_custom_cnn.pth"
)

## 8. Test Metrics and Confusion Matrix

In [ ]:
def get_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data, target in loader:
            pred = model(data.to(device)).argmax(1).cpu()
            y_true.extend(target.numpy())
            y_pred.extend(pred.numpy())
    return np.array(y_true), np.array(y_pred)

y_true, y_pred = get_predictions(final_cnn, test_loader)
cnn_accuracy = accuracy_score(y_true, y_pred)
cnn_precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
cnn_recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
cnn_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
print("Accuracy:", cnn_accuracy)
print("Precision:", cnn_precision)
print("Recall:", cnn_recall)
print("F1:", cnn_f1)

fig, ax = plt.subplots(figsize=(10,10))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names, xticks_rotation=45, ax=ax)
plt.tight_layout(); plt.show()

### Recorded Final Custom CNN Result

- Validation accuracy: **90.96%**
- Test accuracy: **92.07%**
- Weighted precision: **92.37%**
- Weighted recall: **92.07%**
- Weighted F1-score: **92.09%**

## 9. Frozen ResNet18 Comparison

ResNet18 is used as a fixed feature extractor, following the transfer-learning approach from class.

In [ ]:
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in resnet18.parameters():
    param.requires_grad = False
resnet18.fc = nn.Linear(resnet18.fc.in_features, 10)
resnet18 = resnet18.to(device)

torch.manual_seed(random_seed)
resnet18, resnet_history, resnet_best_loss, resnet_best_val_acc = train_model(
    resnet18, train_loader, val_loader, 0.001, RESNET_EPOCHS, PATIENCE, "resnet18_best.pth"
)
resnet_true, resnet_pred = get_predictions(resnet18, test_loader)
print("ResNet18 Accuracy:", accuracy_score(resnet_true, resnet_pred))

### Recorded Model Comparison

| Model | Validation Accuracy | Test Accuracy | Precision | Recall | Weighted F1 |
|---|---:|---:|---:|---:|---:|
| **Final 3-layer Custom CNN** | **90.96%** | **92.07%** | **92.37%** | **92.07%** | **92.09%** |
| Frozen ResNet18 | 81.53% | 82.44% | 82.93% | 82.44% | 82.32% |

## 10. Development and Ablation Findings

The initial course-level run used the same 3-layer CNN but default `RandomResizedCrop` and only four tuning epochs. In the controlled run it reached **85.60%** test accuracy. Restricting the crop to `scale=(0.8,1.0)` and increasing tuning to six epochs raised the result to **92.07%** without making the CNN deeper.

Additional augmentation was then tested separately:

| Augmentation Setting | Test Accuracy | Weighted F1 | Change vs Final Baseline |
|---|---:|---:|---:|
| **Final baseline: crop + horizontal flip** | **92.07%** | **92.09%** | **0.00 pp** |
| + Color Jitter | 91.48% | 91.49% | -0.59 pp |
| + Vertical Flip | 89.83% | 89.95% | -2.25 pp |
| + Rotation pipeline | 67.60% | 67.46% | -24.47 pp |

The rotation experiment also included resize-and-center-crop operations to avoid black borders, so its large decrease is not interpreted as the isolated effect of rotation. It only shows that the tested rotation preprocessing pipeline was not suitable.

Because none of the extra augmentation methods improved the baseline, **VerticalFlip, Rotation, and ColorJitter are not used in the final model**.

## 11. Final Conclusion

The final solution is a **3-layer Custom CNN** using learning rate **0.0005**, batch size **32**, dropout **0.3**, `RandomResizedCrop(scale=(0.8,1.0))`, horizontal flipping, BatchNorm, and early stopping.

The final held-out test accuracy is **92.07%** with a weighted F1-score of **92.09%**. The frozen ResNet18 reached **82.44%** test accuracy.

The experiments show that a simple course-level CNN can perform strongly on EuroSAT when the crop range and hyperparameter selection are chosen carefully. More preprocessing was not automatically better, so the final pipeline keeps the simplest tested augmentation strategy that achieved the best result.